# LinkGuard — URL Safety Classifier Training

**Before running:** Runtime → Change runtime type → T4 GPU

---

## One-time setup

On your local machine:
```bash
cd /home/vanji/Videos/LinkGuard
zip -r linkguard-model.zip linkguard-model/
```
Upload `linkguard-model.zip` to Google Drive root, then run cells below.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/linkguard_training'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive mounted. Output dir:', DRIVE_DIR)

## 2. Extract linkguard-model from Drive

In [ ]:
import os, zipfile

ZIP_PATH  = '/content/drive/MyDrive/linkguard-model.zip'  # change if needed
MODEL_DIR = '/content/linkguard-model'

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        f'Zip not found at {ZIP_PATH}\n'
        'On local machine: zip -r linkguard-model.zip linkguard-model/\n'
        'Then upload to Google Drive and update ZIP_PATH above.'
    )

if os.path.exists(MODEL_DIR):
    print('Already extracted — re-extracting to pick up latest code...')
    import shutil
    shutil.rmtree(MODEL_DIR)

print(f'Extracting {ZIP_PATH}...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('/content')
print('Extracted.')

os.chdir(MODEL_DIR)
print('Working directory:', os.getcwd())
print('Contents:', sorted(os.listdir('.')))

## 3. Install dependencies

In [ ]:
!pip install -q -r requirements.txt
print('Done.')

## 4. Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('No GPU. Go to Runtime → Change runtime type → T4 GPU')

## 5. Download training data

Sources (all free, no auth):
- URLhaus — malware URLs
- OpenPhish — phishing
- ThreatFox — bulk CSV export (no API key)
- Phishing.Database — 400K+ active phishing (GitHub)
- Tranco top-1M — safe domains

In [ ]:
import os, shutil

RAW_SRC  = f'{DRIVE_DIR}/raw'
RAW_DEST = 'data/raw'
os.makedirs(RAW_DEST, exist_ok=True)

# Expected files with the new sources
EXPECTED = {'urlhaus.txt', 'openphish.txt', 'threatfox.txt',
            'phishing_database.txt', 'tranco_top1m.csv'}

cached = set(os.listdir(RAW_SRC)) if os.path.exists(RAW_SRC) else set()
if EXPECTED.issubset(cached):
    print('Raw data found in Drive — restoring...')
    for f in os.listdir(RAW_SRC):
        shutil.copy(f'{RAW_SRC}/{f}', f'{RAW_DEST}/{f}')
    print('Files:', sorted(os.listdir(RAW_DEST)))
else:
    missing = EXPECTED - cached
    if missing:
        print(f'Missing from Drive cache: {missing} — downloading fresh...')
    !python data/collect.py
    os.makedirs(RAW_SRC, exist_ok=True)
    for f in os.listdir(RAW_DEST):
        shutil.copy(f'{RAW_DEST}/{f}', f'{RAW_SRC}/{f}')
    print('Saved to Drive for future runs.')

## 6. Preprocess

Dynamic balancing: safe URLs capped at 2× dangerous count automatically.

In [ ]:
import os, shutil

PROC_SRC  = f'{DRIVE_DIR}/processed'
PROC_DEST = 'data/processed'
os.makedirs(PROC_DEST, exist_ok=True)

# Always re-preprocess if we re-extracted the zip (new code may have changed)
print('Preprocessing...')
!python data/preprocess.py
os.makedirs(PROC_SRC, exist_ok=True)
for f in os.listdir(PROC_DEST):
    shutil.copy(f'{PROC_DEST}/{f}', f'{PROC_SRC}/{f}')
print('Saved to Drive.')

## 7. Train

In [ ]:
import os, shutil

CKPT_SRC  = f'{DRIVE_DIR}/best_model'
CKPT_DEST = 'model_output/best_model'
os.makedirs(CKPT_DEST, exist_ok=True)

if os.path.exists(f'{CKPT_SRC}/model.pt'):
    print('Checkpoint found in Drive — restoring to resume...')
    for f in os.listdir(CKPT_SRC):
        shutil.copy(f'{CKPT_SRC}/{f}', f'{CKPT_DEST}/{f}')

resume_flag = '--resume model_output/best_model/model.pt' if os.path.exists(f'{CKPT_DEST}/model.pt') else ''
print('Mode:', resume_flag or 'fresh training')

In [ ]:
!python train/train.py {resume_flag}

In [ ]:
# Save checkpoint to Drive immediately after training
import shutil, os
os.makedirs(CKPT_SRC, exist_ok=True)
for f in os.listdir(CKPT_DEST):
    shutil.copy(f'{CKPT_DEST}/{f}', f'{CKPT_SRC}/{f}')
print('Checkpoint saved to Drive.')

## 8. Evaluate

In [ ]:
!python train/evaluate.py

In [ ]:
from IPython.display import Image, display
display(Image('model_output/confusion_matrix.png'))
display(Image('model_output/roc_curves.png'))

In [ ]:
with open('model_output/threshold_analysis.txt') as f:
    print(f.read())

## 9. Export to ONNX

In [ ]:
!python train/export_onnx.py

## 10. Save all outputs to Drive

In [ ]:
import shutil, os

ONNX_DRIVE = f'{DRIVE_DIR}/onnx'
os.makedirs(ONNX_DRIVE, exist_ok=True)
for f in os.listdir('model_output/onnx'):
    shutil.copy(f'model_output/onnx/{f}', f'{ONNX_DRIVE}/{f}')

EVAL_DRIVE = f'{DRIVE_DIR}/eval'
os.makedirs(EVAL_DRIVE, exist_ok=True)
for fname in ['confusion_matrix.png', 'roc_curves.png',
              'threshold_analysis.txt', 'classification_report.txt',
              'training_log.csv']:
    src = f'model_output/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'{EVAL_DRIVE}/{fname}')

print('All outputs saved to Drive:')
for d in sorted(os.listdir(DRIVE_DIR)):
    sub = f'{DRIVE_DIR}/{d}'
    if os.path.isdir(sub):
        print(f'  {d}/  {os.listdir(sub)}')

## 11. Quick inference test

In [ ]:
!python serve/local_inference.py

## Done!

Your model is in Drive at `linkguard_training/onnx/model_int8.onnx`.

Next: deploy `serve/spaces/` to HuggingFace Spaces → set `LG_MODEL_URL` in `background.js`.